# LNA 示例让我们使用 Infineon 的 BFU520 晶体管设计一个 LNA。首先，我们需要导入 scikit-rf 和其他一些实用工具：

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import skrf as rf
from skrf.media import DistributedCircuit
from skrf.network import s2z

plt.rcParams['figure.figsize'] = [10, 10]

f = rf.Frequency(0.4, 2, 101, 'GHz')
tem = DistributedCircuit(f, z0=50)

In [ ]:
# import the scattering parameters/noise data for the transistor
bjt = rf.Network('BFU520_05V0_010mA_NF_SP.s2p').interpolate(f)
print(bjt)

让我们绘制它的史密斯圆图：

In [ ]:
bjt.plot_s_smith(lw=2)

现在，让我们计算源阻抗和负载阻抗的稳定性曲线。我稍微滥用了 `Network` 类型来绘制曲线；通常，传递给 `Network` 的曲线应该是一个频率的函数，但只要不尝试对这些曲线应用其他函数，绘制这些圆也是可行的。

In [ ]:
sqabs = lambda x: np.square(np.absolute(x))  # noqa: E731

delta = bjt.s11.s*bjt.s22.s - bjt.s12.s*bjt.s21.s
rl = np.absolute((bjt.s12.s * bjt.s21.s)/(sqabs(bjt.s22.s) - sqabs(delta)))
cl = np.conj(bjt.s22.s - delta*np.conj(bjt.s11.s))/(sqabs(bjt.s22.s) - sqabs(delta))

rs = np.absolute((bjt.s12.s * bjt.s21.s)/(sqabs(bjt.s11.s) - sqabs(delta)))
cs = np.conj(bjt.s11.s - delta*np.conj(bjt.s22.s))/(sqabs(bjt.s11.s) - sqabs(delta))

def calc_circle(c, r):
    theta = np.linspace(0, 2*np.pi, 1000)
    return c + r*np.exp(1.0j*theta)

In [ ]:
for i, f in enumerate(bjt.f):
    # decimate it a little
    if i % 100 != 0:
        continue
    n = rf.Network(name=str(f/1.e+9), s=calc_circle(cs[i][0, 0], rs[i][0, 0]))
    n.plot_s_smith()

In [ ]:
for i, f in enumerate(bjt.f):
    # decimate it a little
    if i % 100 != 0:
        continue
    n = rf.Network(name=str(f/1.e+9), s=calc_circle(cl[i][0, 0], rl[i][0, 0]))
    n.plot_s_smith()

因此，我们可以看到，我们需要避免在输入匹配网络中的短路附近放置电感负载，并在输出端避免放置高阻抗电感负载。让我们绘制一些恒定噪声圆。首先，我们从网络模型中获取目标频率的噪声参数：

In [ ]:
idx_915mhz = rf.util.find_nearest_index(bjt.f, 915.e+6)

# we need the normalized equivalent noise and optimum source coefficient to calculate the constant noise circles
rn = bjt.rn[idx_915mhz]/50
gamma_opt = bjt.g_opt[idx_915mhz]
fmin = bjt.nfmin[idx_915mhz]

for nf_added in [0, 0.1, 0.2, 0.5]:
    nf = 10**(nf_added/10) * fmin

    N = (nf - fmin)*abs(1+gamma_opt)**2/(4*rn)
    c_n = gamma_opt/(1+N)
    r_n = 1/(1-N)*np.sqrt(N**2 + N*(1-abs(gamma_opt)**2))

    n = rf.Network(name=str(nf_added), s=calc_circle(c_n, r_n))
    n.plot_s_smith()

print("the optimum source reflection coefficient is ", gamma_opt)

从图表中我们可以看到，将输入阻抗保持在 50 欧姆，额外的噪声可以控制在 0.1 dB 以下，这似乎是一个不错的结果。我实际上不太确定这些数值是否与我列出的噪声系数增量对应，但这些圆圈至少应该代表噪声系数的增加。所以，让我们将输入阻抗保持在 50 欧姆，并确定如何匹配输出网络，以最大限度地提高增益和稳定性。让我们看看将负载阻抗与未匹配的输入阻抗进行匹配会产生什么结果：

In [ ]:
gamma_s = 0.0

gamma_l = np.conj(bjt.s22.s - bjt.s21.s*gamma_s*bjt.s12.s/(1-bjt.s11.s*gamma_s))
gamma_l = gamma_l[idx_915mhz, 0, 0]
is_gamma_l_stable = np.absolute(gamma_l - cl[idx_915mhz]) > rl[idx_915mhz]

gamma_l, is_gamma_l_stable

这似乎与负载不稳定性曲线非常接近，因此选择具有较低增益的负载点以提高稳定性，或者选择具有更多噪声的不同源阻抗可能是有意义的。但现在，我们先构建一个匹配网络，看看它的性能如何：

In [ ]:
def calc_matching_network_vals(z1, z2):
    flipped = np.real(z1) < np.real(z2)
    if flipped:
        z2, z1 = z1, z2

    # cancel out the imaginary parts of both input and output impedances
    z1_par = 0.0
    if abs(np.imag(z1)) > 1e-6:
        # parallel something to cancel out the imaginary part of
        # z1's impedance
        z1_par = 1/(-1j*np.imag(1/z1))
        z1 = 1/(1./z1 + 1/z1_par)
    z2_ser = 0.0
    if abs(np.imag(z2)) > 1e-6:
        z2_ser = -1j*np.imag(z2)
        z2 = z2 + z2_ser

    Q = np.sqrt((np.real(z1) - np.real(z2))/np.real(z2))
    x1 = -1.j * np.real(z1)/Q
    x2 = 1.j * np.real(z2)*Q

    x1_tot = 1/(1/z1_par + 1/x1)
    x2_tot = z2_ser + x2
    if flipped:
        return x2_tot, x1_tot
    else:
        return x1_tot, x2_tot

z_l = s2z(np.array([[[gamma_l]]]))[0,0,0]
# note that we're matching against the conjugate;
# this is because we want to see z_l from the BJT side
# if we plugged in z the matching network would make
# the 50 ohms look like np.conj(z) to match against it, so
# we use np.conj(z_l) so that it'll look like z_l from the BJT's side
z_par, z_ser = calc_matching_network_vals(np.conj(z_l), 50)
z_l, z_par, z_ser

让我们计算一下元件的数值：

In [ ]:
c_par = np.real(1/(2j*np.pi*915e+6*z_par))
l_ser = np.real(z_ser/(2j*np.pi*915e+6))

print(c_par, l_ser)

电容值有点低，但电感值似乎是合理的。让我们来测试一下：

In [ ]:
output_network = tem.shunt_capacitor(c_par) ** tem.inductor(l_ser)

amplifier = bjt ** output_network

amplifier.plot_s_smith()

这看起来相当合理；让我们查看 S21，看看我们得到了什么：

In [ ]:
amplifier.s21.plot_s_db()

所以，增益约为 18 dB；让我们看看它的噪声系数是多少：

In [ ]:
10*np.log10(amplifier.nf(50.)[idx_915mhz])

因此，噪声系数为 0.96 dB，这与 BJT 理想电路的最佳噪声系数 0.95 dB 相当接近。